### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [ ]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748])

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


**1. Vectorizar documentos**

In [2]:
# Tomamos el corpus de bbc news y seleccionaremos 5 articulos
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import zipfile
import os
import numpy as np

np.random.seed(42)

url = "http://mlg.ucd.ie/files/datasets/bbc-fulltext.zip"

r = requests.get(url)

with open("bbc.zip", "wb") as f:
    f.write(r.content)

# Descomprimir
with zipfile.ZipFile("bbc.zip", "r") as zip_ref:
    zip_ref.extractall(".")

# Cargar el corpus
bbc = load_files(
    "bbc",
    encoding="latin1",
    decode_error="replace"
)

# Ver qué carpetas se generaron
print(os.listdir("."))

# Vectorizamos todos los documentos
vectorizer = TfidfVectorizer(
    stop_words="english"
)

X = vectorizer.fit_transform(bbc.data)

# Elegimos 5 documentos al azar
np.random.seed(42)

indices_5 = np.random.choice(
    len(bbc.data),
    size=5,
    replace=False
)

print("Documentos seleccionados:", indices_5)

# Del corpus elegido de la bbc news elegimos 5 articulos al azar.

for indice in indices_5:

    # Similaridad del documento elegido contra TODOS
    similitudes = cosine_similarity(
        X[indice],
        X
    ).flatten()

    # Evitamos que él mismo aparezca como más similar
    similitudes[indice] = -1

    # Índices de los 5 más similares
    indices_similares = np.argsort(
        similitudes
    )[-5:][::-1]

    print("=" * 80)
    print("DOCUMENTO SELECCIONADO:", indice)
    print("CLASE:", bbc.target_names[bbc.target[indice]])

    print("\n5 DOCUMENTOS MÁS SIMILARES:")

    for j in indices_similares:

        print(
            f"Documento: {j} | "
            f"Clase: {bbc.target_names[bbc.target[j]]} | "
            f"Similaridad: {similitudes[j]:.4f}"
        )








['.config', 'bbc', 'bbc.zip', 'sample_data']
Documentos seleccionados: [ 414  420 1644  416 1232]
DOCUMENTO SELECCIONADO: 414
CLASE: entertainment

5 DOCUMENTOS MÁS SIMILARES:
Documento: 1172 | Clase: tech | Similaridad: 0.2797
Documento: 1360 | Clase: tech | Similaridad: 0.2780
Documento: 365 | Clase: entertainment | Similaridad: 0.2659
Documento: 445 | Clase: entertainment | Similaridad: 0.2441
Documento: 881 | Clase: entertainment | Similaridad: 0.2322
DOCUMENTO SELECCIONADO: 420
CLASE: entertainment

5 DOCUMENTOS MÁS SIMILARES:
Documento: 603 | Clase: entertainment | Similaridad: 0.4300
Documento: 1368 | Clase: entertainment | Similaridad: 0.4148
Documento: 1616 | Clase: entertainment | Similaridad: 0.2872
Documento: 1296 | Clase: entertainment | Similaridad: 0.2838
Documento: 1682 | Clase: entertainment | Similaridad: 0.2824
DOCUMENTO SELECCIONADO: 1644
CLASE: business

5 DOCUMENTOS MÁS SIMILARES:
Documento: 790 | Clase: business | Similaridad: 0.5929
Documento: 606 | Clase: busin

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    bbc.data,
    bbc.target,
    test_size=0.20,
    random_state=42,
    stratify=bbc.target
)

print("Documentos de entrenamiento:", len(X_train))
print("Documentos de test:", len(X_test))
print("Categorías:", bbc.target_names)


# --------------------------------------------------
# Vectorización
# --------------------------------------------------

vectorizador = TfidfVectorizer(
    stop_words="english"
)

X_train_tfidf = vectorizador.fit_transform(X_train)
X_test_tfidf = vectorizador.transform(X_test)


# --------------------------------------------------
# Compararamos cada documento de test
# con todos los documentos de entrenamiento
# --------------------------------------------------

similitudes = cosine_similarity(
    X_test_tfidf,
    X_train_tfidf
)


# --------------------------------------------------
# Buscar el documento de entrenamiento
# más similar para cada documento de test
# --------------------------------------------------

indices_mas_similares = np.argmax(
    similitudes,
    axis=1
)


# --------------------------------------------------
# Asignar al documento de test
# la clase del documento más similar
# --------------------------------------------------

y_pred = y_train[indices_mas_similares]


print("Clasificación realizada.")
print("Cantidad de documentos clasificados:", len(y_pred))


Documentos de entrenamiento: 1780
Documentos de test: 445
Categorías: ['business', 'entertainment', 'politics', 'sport', 'tech']
Clasificación realizada.
Cantidad de documentos clasificados: 445


**Punto 3**

In [5]:
from sklearn.naive_bayes import MultinomialNB,ComplementNB
from sklearn.metrics import f1_score
import pandas as pd


resultados = []

# Valores que vamos a probar
valores_min_df = [1, 2, 5]
valores_alpha = [0.1, 0.5, 1.0]

for min_df in valores_min_df:

    # Vectorizador
    vectorizador = TfidfVectorizer(
        stop_words="english",
        min_df=min_df
    )

    X_train_tfidf = vectorizador.fit_transform(X_train)
    X_test_tfidf = vectorizador.transform(X_test)

    # -------------------------
    # MultinomialNB
    # -------------------------

    for alpha in valores_alpha:

        modelo = MultinomialNB(alpha=alpha)
        modelo.fit(X_train_tfidf, y_train)
        y_pred = modelo.predict(X_test_tfidf)
        f1 = f1_score(
            y_test,
            y_pred,
            average="macro"
        )

        resultados.append({
            "Modelo": "MultinomialNB",
            "min_df": min_df,
            "alpha": alpha,
            "F1 Macro": f1
        })

    # -------------------------
    # ComplementNB
    # -------------------------

    for alpha in valores_alpha:

        modelo = ComplementNB(alpha=alpha)
        modelo.fit(X_train_tfidf, y_train)
        y_pred = modelo.predict(X_test_tfidf)
        f1 = f1_score(
            y_test,
            y_pred,
            average="macro"
        )

        resultados.append({
            "Modelo": "ComplementNB",
            "min_df": min_df,
            "alpha": alpha,
            "F1 Macro": f1
        })

        ## Observamos los resultados obtenidos ##


resultados_df = pd.DataFrame(resultados)
resultados_df = resultados_df.sort_values(
    by="F1 Macro",
    ascending=False
)

resultados_df


,Modelo,min_df,alpha,F1 Macro
10,ComplementNB,2,0.5,0.967791
4,ComplementNB,1,0.5,0.967791
16,ComplementNB,5,0.5,0.967791
8,MultinomialNB,2,1.0,0.967499
11,ComplementNB,2,1.0,0.965613
17,ComplementNB,5,1.0,0.965613
5,ComplementNB,1,1.0,0.965613
3,ComplementNB,1,0.1,0.965587
9,ComplementNB,2,0.1,0.965587
15,ComplementNB,5,0.1,0.965270


**4. Transponer la matriz documento-término.**

In [6]:
# --------------------------------------------------
# 4. Transponer la matriz documento-término.
# --------------------------------------------------

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Transponer la matriz documento-término
X_term_doc = X_train_tfidf.T

# Obtener los términos correspondientes a las filas
terminos = vectorizador.get_feature_names_out()

print("Matriz documento-término:", X_train_tfidf.shape)
print("Matriz término-documento:", X_term_doc.shape)

# Elegimos manualmente 5 palabras interpretables
palabras_elegidas = [
    "government",
    "football",
    "music",
    "computer",
    "company"
]


# Analizar cada palabra
for palabra in palabras_elegidas:

    # Verificar que la palabra exista
    if palabra not in terminos:
        print("La palabra", palabra, "no está en el vocabulario.")
        continue

    # Índice de la palabra
    indice_palabra = np.where(terminos == palabra)[0][0]

    # Similaridad con todas las palabras
    similitudes = cosine_similarity(
        X_term_doc[indice_palabra],
        X_term_doc
    ).flatten()

    # No consideramos la palabra consigo misma
    similitudes[indice_palabra] = -1

    # Obtener las 5 más similares
    indices_similares = np.argsort(
        similitudes
    )[-5:][::-1]

    print("\n" + "=" * 70)
    print("PALABRA:", palabra)
    print("=" * 70)

    for indice in indices_similares:

        print(
            f"{terminos[indice]:20s}"
            f" Similaridad: {similitudes[indice]:.4f}"
        )

Matriz documento-término: (1780, 7964)
Matriz término-documento: (7964, 1780)

PALABRA: government
said                 Similaridad: 0.4238
minister             Similaridad: 0.4147
public               Similaridad: 0.3895
mr                   Similaridad: 0.3675
plans                Similaridad: 0.3661

PALABRA: football
manchester           Similaridad: 0.3135
boss                 Similaridad: 0.3071
club                 Similaridad: 0.2955
premiership          Similaridad: 0.2938
league               Similaridad: 0.2910

PALABRA: music
itunes               Similaridad: 0.4346
downloading          Similaridad: 0.3972
songs                Similaridad: 0.3795
napster              Similaridad: 0.3673
albums               Similaridad: 0.3606

PALABRA: computer
computers            Similaridad: 0.4168
software             Similaridad: 0.4145
mouse                Similaridad: 0.3696
pc                   Similaridad: 0.3168
mac                  Similaridad: 0.3096

PALABRA: company
firm     

**Conclusiones Generales**


A partir de los experimentos realizados sobre el corpus BBC News, se pudo observar cómo distintas técnicas de representación vectorial permiten analizar tanto la similitud entre documentos como la similitud entre términos.

En primer lugar, la representación de los documentos mediante TF-IDF permitió transformar los textos en vectores numéricos y utilizar la similitud coseno para comparar su contenido. Al seleccionar documentos al azar y buscar sus documentos más similares, se observó que los resultados generalmente presentaban contenido temáticamente relacionado. En muchos casos, los documentos más similares pertenecían a la misma categoría, lo que muestra que la representación TF-IDF logra capturar parte de la estructura temática del corpus.

Luego se implementó un modelo de clasificación por prototipos. Para cada documento del conjunto de test se calculó su similitud con todos los documentos de entrenamiento y se asignó la clase correspondiente al documento más similar. Esto permitió realizar la clasificación sin entrenar un modelo de clasificación tradicional. Sin embargo, al basarse únicamente en un único documento vecino, sus predicciones pueden verse afectadas por documentos particulares o casos ambiguos. El experimento permitió establecer además una referencia con la cual comparar los modelos de Naïve Bayes.

Posteriormente se entrenaron modelos MultinomialNB y ComplementNB, evaluándolos mediante F1-Score Macro. Se realizaron diferentes experimentos modificando parámetros del vectorizador y de los modelos, manteniendo fijo el hiperparámetro "ngram_range", tal como establecía la consigna. Los resultados mostraron que la elección de los parámetros de representación y del modelo tiene influencia directa sobre el desempeño de clasificación. Por lo tanto, no existe una única configuración que deba considerarse independientemente del corpus: es necesario experimentar y comparar las distintas alternativas mediante una métrica común.

El uso de F1 Macro fue útil porque permite considerar el desempeño sobre cada categoría y no solamente el porcentaje total de documentos correctamente clasificados. De esta manera, la evaluación resulta más representativa cuando se quiere analizar el comportamiento del clasificador sobre las diferentes clases del corpus.

Finalmente, al transponer la matriz documento-término se obtuvo una matriz término-documento, permitiendo interpretar cada palabra como un vector que representa su distribución en los documentos. Al calcular la similitud entre palabras, se encontraron términos que presentan patrones de aparición similares dentro del corpus. Estas similitudes no necesariamente representan sinonimia, sino que indican que las palabras tienden a aparecer en documentos o contextos temáticamente relacionados.

